In [1]:
# ==============================================================================
# STELLAR-ARM 4-WAY OUT-OF-DISTRIBUTION AUDIT: OFFICIAL RM-BENCH
# Evaluates SCALAR_RM, INDEPENDENT_6D, STELLAR_NO_VETO, and STELLAR_FULL
# using ONLY scalar_logit (actual RM reward) for apples-to-apples comparison.
# ==============================================================================
!pip install -q --upgrade "transformers>=4.44.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "accelerate>=0.33.0" "datasets>=2.20.0"

import os
import gc
import time
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Dict, List, Any
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, set_peft_model_state_dict

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
# ------------------------------------------------------------------------------
# 1. AUTO-DISCOVER BASE MODEL & ALL 4 CHECKPOINTS
# ------------------------------------------------------------------------------
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
if not os.path.exists(MODEL_PATH):
    for root, _, files in os.walk("/kaggle/input"):
        if "config.json" in files and "qwen" in root.lower() and "stellar" not in root.lower():
            MODEL_PATH = root
            break
print(f"[BASE MODEL] {MODEL_PATH}")

CHECKPOINTS = {
    "SCALAR_RM": {
        "keywords": ["scalar_rm", "scalar-rm"],
        "default_dir": "/kaggle/input/notebooks/pradeep01223/ablations-stellar-scalar-rm/checkpoints/SCALAR_RM/best",
        "associative": False, "use_dag_mixer": False, "use_monotonic": False,
        "use_lrsv_veto": False, "scalar_only_mode": True,
    },
    "INDEPENDENT_6D": {
        "keywords": ["independent_6d", "independent-6d"],
        "default_dir": "/kaggle/input/notebooks/mrsstrange012/ablations-stellar-arm-independent-6d/checkpoints/INDEPENDENT_6D/best",
        "associative": False, "use_dag_mixer": False, "use_monotonic": False,
        "use_lrsv_veto": False, "scalar_only_mode": False,
    },
    "STELLAR_NO_VETO": {
        "keywords": ["stellar_no_veto", "no_veto", "no-veto"],
        "default_dir": "/kaggle/input/notebooks/mrsstrange012/ablations-stellar-arm-no-veto/checkpoints/STELLAR_NO_VETO/best",
        "associative": True, "use_dag_mixer": True, "use_monotonic": True,
        "use_lrsv_veto": False, "scalar_only_mode": False,
    },
    "STELLAR_FULL": {
        "keywords": ["stellar_full", "stellar-shard1", "shard1"],
        "default_dir": "/kaggle/input/notebooks/rjjack0112/stellar-shard1-training-pipeline/checkpoints/STELLAR_FULL/best",
        "associative": True, "use_dag_mixer": True, "use_monotonic": True,
        "use_lrsv_veto": True, "scalar_only_mode": False,
    },
}

def resolve_checkpoint_dir(mode_name: str, cfg: Dict) -> str:
    d = cfg["default_dir"]
    if os.path.exists(os.path.join(d, "lora_weights.pt")) and os.path.exists(os.path.join(d, "arm_heads.pt")):
        return d
    candidates = []
    for search_root in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.exists(search_root):
            continue
        for root, _, files in os.walk(search_root):
            if "lora_weights.pt" in files and "arm_heads.pt" in files:
                if any(kw in root.lower() for kw in cfg["keywords"]):
                    candidates.append(root)
    if candidates:
        candidates.sort(key=lambda p: (0 if p.rstrip("/").endswith("best") else 1, len(p)))
        return candidates[0]
    raise FileNotFoundError(f"Missing checkpoint for {mode_name}!")

for name, cfg in CHECKPOINTS.items():
    cfg["dir"] = resolve_checkpoint_dir(name, cfg)
    print(f"[CKPT] {name:<16} -> {cfg['dir']}")

[BASE MODEL] /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
[CKPT] SCALAR_RM        -> /kaggle/input/notebooks/pradeep01223/ablations-stellar-scalar-rm/checkpoints/SCALAR_RM/best
[CKPT] INDEPENDENT_6D   -> /kaggle/input/notebooks/mrsstrange012/ablations-stellar-arm-independent-6d/checkpoints/INDEPENDENT_6D/best
[CKPT] STELLAR_NO_VETO  -> /kaggle/input/notebooks/mrsstrange012/ablations-stellar-arm-no-veto/checkpoints/STELLAR_NO_VETO/best
[CKPT] STELLAR_FULL     -> /kaggle/input/notebooks/rjjack0112/stellar-shard1-training-pipeline/checkpoints/STELLAR_FULL/best


In [3]:
# ------------------------------------------------------------------------------
# 2. LOAD 4-BIT QWEN3-8B PEFT BACKBONE ONCE
# ------------------------------------------------------------------------------
num_gpus = torch.cuda.device_count()
assert num_gpus > 0, "No GPU detected! Enable 2x T4 GPU in Kaggle settings."
HEAD_DEVICE = torch.device(f"cuda:{num_gpus - 1}") if num_gpus > 1 else torch.device("cuda:0")
MAX_LENGTH = 1024
BATCH_SIZE = 4  # Safe for 1024-token tail batches on 2x T4

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|endoftext|>"
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
max_mem = {0: "8.0GiB", 1: "12.0GiB"} if num_gpus >= 2 else {0: "14.5GiB"}

base_model = AutoModel.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
    max_memory=max_mem,
    attn_implementation="eager",
    trust_remote_code=True,
)
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
peft_backbone = get_peft_model(base_model, lora_config)
peft_backbone.eval()
print(f"[BACKBONE READY] Loaded on {num_gpus} GPU(s) | Head device: {HEAD_DEVICE}", flush=True)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BACKBONE READY] Loaded on 2 GPU(s) | Head device: cuda:1


In [4]:
# ------------------------------------------------------------------------------
# 3. ARCHITECTURE DEFINITION (EXACT MATCH TO SAVED CHECKPOINTS)
# ------------------------------------------------------------------------------
class CausalDAGLayer(nn.Module):
    def __init__(self, d_model: int = 512, nhead: int = 8, dim_feedforward: int = 1024, dropout: float = 0.05):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.gate_proj = nn.Linear(d_model * 2, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(dim_feedforward, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, associative: bool = True) -> torch.Tensor:
        dev = z.device
        attn_mask = (
            torch.triu(torch.ones((6, 6), dtype=torch.bool, device=dev), diagonal=1)
            if associative else ~torch.eye(6, dtype=torch.bool, device=dev)
        )
        c, _ = self.self_attn(z, z, z, attn_mask=attn_mask, need_weights=False)
        g = torch.sigmoid(self.gate_proj(torch.cat([z, c], dim=-1)))
        z_mixed = (1.0 - g) * z + g * c
        z_mixed = torch.cat([z[:, :1, :], z_mixed[:, 1:, :]], dim=1)
        z_out = self.norm1(z_mixed)
        z_out = self.norm2(z_out + self.ffn(z_out))
        return torch.cat([z[:, :1, :], z_out[:, 1:, :]], dim=1)


class Qwen3AssociativeRM(nn.Module):
    GATEKEEPER_INDICES = [0, 3, 4]

    def __init__(self, peft_backbone: nn.Module, associative=True, use_monotonic=True,
                 use_lrsv_veto=True, scalar_only_mode=False, dag_num_layers=2,
                 dag_hidden_size=512, dag_num_heads=8, beta=1.0,
                 veto_kappa=1.5, monotonic_tau=0.5, monotonic_delta_tol=0.20):
        super().__init__()
        self.backbone = peft_backbone
        self.associative = associative
        self.use_monotonic = use_monotonic
        self.use_lrsv_veto = use_lrsv_veto
        self.scalar_only_mode = scalar_only_mode
        self.beta = beta
        self.veto_kappa = veto_kappa
        self.monotonic_tau = monotonic_tau
        self.monotonic_delta_tol = monotonic_delta_tol

        self.input_device = self.backbone.get_input_embeddings().weight.device
        self.head_device = HEAD_DEVICE
        self.hidden_size = self.backbone.config.hidden_size

        self.query_embeds = nn.Parameter(torch.zeros(1, 6, self.hidden_size, dtype=torch.float32, device=self.head_device))
        self.dag_attn_bias = nn.Parameter(torch.zeros((6, 6), dtype=torch.float32, device=self.head_device))
        prior = torch.tril(torch.ones((6, 6), dtype=torch.float32, device=self.head_device), diagonal=-1)
        prior[4, 3], prior[4, 2], prior[3, 2] = 2.5, 2.0, 1.5
        self.register_buffer("dag_prior", prior, persistent=False)

        self.dag_down_proj = nn.Linear(self.hidden_size, dag_hidden_size, dtype=torch.float32, device=self.head_device)
        self.dag_norm = nn.LayerNorm(dag_hidden_size, dtype=torch.float32, device=self.head_device)
        self.dag_layers = nn.ModuleList([
            CausalDAGLayer(dag_hidden_size, dag_num_heads, dag_hidden_size * 2, dropout=0.05).to(
                device=self.head_device, dtype=torch.float32
            ) for _ in range(dag_num_layers)
        ])
        self.dim_heads = nn.ModuleList([
            nn.Linear(dag_hidden_size, 1, dtype=torch.float32, device=self.head_device) for _ in range(6)
        ])
        self.agg_logits = nn.Parameter(torch.zeros(6, dtype=torch.float32, device=self.head_device))
        self.veto_thresholds = nn.Parameter(torch.zeros(len(self.GATEKEEPER_INDICES), dtype=torch.float32, device=self.head_device))
        self.veto_raw_lambda = nn.Parameter(torch.tensor(1.85, dtype=torch.float32, device=self.head_device))

    def build_4d_attention_mask(self, token_mask: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
        B, L = token_mask.shape
        T = L + 6
        dev = token_mask.device
        causal_grid = torch.tril(torch.ones((T, T), dtype=torch.bool, device=dev))
        if not self.associative:
            causal_grid[L:, L:] = torch.eye(6, dtype=torch.bool, device=dev)
        query_padding = torch.ones((B, 6), dtype=token_mask.dtype, device=dev)
        full_mask_2d = torch.cat([token_mask, query_padding], dim=1)
        valid_attn = causal_grid.unsqueeze(0).unsqueeze(0) & full_mask_2d.bool().unsqueeze(1).unsqueeze(2)
        mask_4d = torch.where(valid_attn, torch.tensor(0.0, dtype=dtype, device=dev), torch.tensor(-1e4, dtype=dtype, device=dev))

        if self.associative and (not self.scalar_only_mode):
            L_b = token_mask.float().sum(dim=-1, keepdim=True).clamp(min=1.0)
            log_L = 0.5 * torch.log(L_b).view(B, 1, 1, 1).to(device=dev, dtype=torch.float32)
            strict_tril = torch.tril(torch.ones((6, 6), dtype=torch.bool, device=dev), diagonal=-1)
            gamma_sig = torch.sigmoid(self.dag_attn_bias.to(device=dev, dtype=torch.float32))
            prior_dev = self.dag_prior.to(device=dev, dtype=torch.float32)
            q_bias = log_L * gamma_sig.unsqueeze(0).unsqueeze(0) + prior_dev.unsqueeze(0).unsqueeze(0)
            q_bias = torch.where(strict_tril.unsqueeze(0).unsqueeze(0), q_bias, torch.zeros_like(q_bias))
            mask_4d[:, :, L:, L:] = mask_4d[:, :, L:, L:] + q_bias.to(dtype=dtype)
        return mask_4d

    def forward_dag_head_only(self, last_6_hidden_fp32: torch.Tensor) -> torch.Tensor:
        z = self.dag_norm(self.dag_down_proj(last_6_hidden_fp32))
        for layer in self.dag_layers:
            z = layer(z, associative=self.associative)
        raw_z = torch.cat([self.dim_heads[i](z[:, i, :]) for i in range(6)], dim=-1)
        z_clamped = torch.clamp(raw_z, min=-15.0, max=15.0)

        if self.scalar_only_mode:
            agg_weights = F.softmax(self.agg_logits, dim=0)
            return (z_clamped * agg_weights).sum(dim=-1)

        if self.associative and self.use_monotonic:
            z3, z4, tau_m = z_clamped[:, 2:3], z_clamped[:, 3:4], self.monotonic_tau
            z_prereq = -tau_m * torch.logsumexp(torch.cat([-z3 / tau_m, -z4 / tau_m], dim=-1), dim=-1, keepdim=True) + tau_m * math.log(2.0)
            z5_mono = torch.clamp(z_prereq + self.monotonic_delta_tol - F.softplus(-raw_z[:, 4:5]), min=-15.0, max=15.0)
            dim_logits = torch.cat([z_clamped[:, :4], z5_mono, z_clamped[:, 5:]], dim=-1)
        else:
            dim_logits = z_clamped

        agg_weights = F.softmax(self.agg_logits, dim=0)
        linear_term = self.beta * (dim_logits * agg_weights).sum(dim=-1)
        if self.use_lrsv_veto:
            gk_logits = dim_logits[:, self.GATEKEEPER_INDICES]
            veto_lambda = F.softplus(self.veto_raw_lambda)
            exponents = -self.veto_kappa * (gk_logits - self.veto_thresholds.unsqueeze(0))
            zero_pad = torch.zeros((exponents.size(0), 1), dtype=exponents.dtype, device=exponents.device)
            veto_penalty = (veto_lambda / self.veto_kappa) * torch.logsumexp(torch.cat([zero_pad, exponents], dim=-1), dim=-1)
            return linear_term - veto_penalty
        return linear_term

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B = input_ids.shape[0]
        input_ids = input_ids.to(self.input_device)
        attention_mask = attention_mask.to(self.input_device)
        tok_embeds = self.backbone.get_input_embeddings()(input_ids)
        q_exp = self.query_embeds.to(device=self.input_device, dtype=tok_embeds.dtype).expand(B, -1, -1)
        inputs_embeds = torch.cat([tok_embeds, q_exp], dim=1)
        full_mask_2d = torch.cat([attention_mask, torch.ones((B, 6), dtype=attention_mask.dtype, device=self.input_device)], dim=1)
        pos_ids = (full_mask_2d.long().cumsum(-1) - 1).clamp(min=0)
        mask_4d = self.build_4d_attention_mask(attention_mask, dtype=tok_embeds.dtype)

        outputs = self.backbone(inputs_embeds=inputs_embeds, attention_mask=mask_4d, position_ids=pos_ids, return_dict=True)
        last_6_fp32 = outputs.last_hidden_state[:, -6:, :].to(device=self.head_device, dtype=torch.float32)
        with torch.cuda.device(self.head_device):
            return self.forward_dag_head_only(last_6_fp32)

    def load_head_state_dict(self, state: Dict[str, Any]):
        self.query_embeds.data.copy_(state["query_embeds"].to(self.head_device))
        self.agg_logits.data.copy_(state["agg_logits"].to(self.head_device))
        if "dag_attn_bias" in state:
            self.dag_attn_bias.data.copy_(state["dag_attn_bias"].to(self.head_device))
            self.dag_down_proj.load_state_dict(state["dag_down_proj"])
            self.dag_norm.load_state_dict(state["dag_norm"])
            if len(self.dag_layers) > 0 and "dag_layers" in state:
                self.dag_layers.load_state_dict(state["dag_layers"])
            self.dim_heads.load_state_dict(state["dim_heads"])
            self.veto_thresholds.data.copy_(state["veto_thresholds"].to(self.head_device))
            self.veto_raw_lambda.data.copy_(state["veto_raw_lambda"].to(self.head_device))
        self.associative = state.get("associative", self.associative)
        self.use_monotonic = state.get("use_monotonic", self.use_monotonic)
        self.use_lrsv_veto = state.get("use_lrsv_veto", self.use_lrsv_veto)
        self.beta = state.get("beta", self.beta)
        self.scalar_only_mode = state.get("scalar_only_mode", self.scalar_only_mode)

In [5]:
# ------------------------------------------------------------------------------
# 4. LOAD RM-BENCH & PRE-TOKENIZE ALL 6 STYLE VARIANTS ONCE
# ------------------------------------------------------------------------------
print("[RM-BENCH] Downloading THU-KEG/RM-Bench...", flush=True)
ds = load_dataset("THU-KEG/RM-Bench", split="train")
n_prompts = len(ds)
domains = [row["domain"] for row in ds]
print(f"[RM-BENCH] Loaded {n_prompts} prompts ({n_prompts * 6} total sequences).", flush=True)

def truncate_prompt_response(prompt: str, response: str, tok, max_len=1024, resp_reserve=640) -> List[int]:
    p_ids = tok.encode(prompt, add_special_tokens=False)
    r_ids = tok.encode(response, add_special_tokens=False)
    if len(p_ids) + len(r_ids) <= max_len:
        return p_ids + r_ids
    r_keep = r_ids[-min(len(r_ids), resp_reserve):]
    p_keep = p_ids[-(max_len - len(r_keep)):] if (max_len - len(r_keep)) > 0 else []
    return p_keep + r_keep

print("[TOKENIZING] Pre-tokenizing all RM-Bench sequences once...", flush=True)
all_seqs = []
for row in ds:
    p = row["prompt"]
    # 3 chosen style levels (0=concise, 1=detailed_plain, 2=detailed_markdown) + 3 rejected style levels
    for c in row["chosen"]:
        all_seqs.append(truncate_prompt_response(p, c, tokenizer, MAX_LENGTH, 640))
    for r in row["rejected"]:
        all_seqs.append(truncate_prompt_response(p, r, tokenizer, MAX_LENGTH, 640))
print(f"[TOKENIZING] Done ({len(all_seqs)} sequences).", flush=True)

@torch.inference_mode()
def score_all_rmbench(rm_model, seqs: List[List[int]], label: str = "", batch_size: int = BATCH_SIZE) -> np.ndarray:
    rm_model.eval()
    n = len(seqs)
    order = np.argsort([len(s) for s in seqs])
    rewards = np.zeros(n, dtype=np.float64)
    pad_id = tokenizer.pad_token_id
    t0 = time.time()

    for start in range(0, n, batch_size):
        idx_batch = order[start:start + batch_size]
        batch_seqs = [seqs[i] for i in idx_batch]
        m_len = max(len(s) for s in batch_seqs)

        ids = torch.full((len(batch_seqs), m_len), pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch_seqs), m_len), dtype=torch.long)
        for row, s in enumerate(batch_seqs):
            ids[row, m_len - len(s):] = torch.tensor(s, dtype=torch.long)
            mask[row, m_len - len(s):] = 1

        rewards[idx_batch] = rm_model(ids, mask).detach().cpu().numpy()

        done = min(start + batch_size, n)
        if (start // batch_size) % 100 == 0 or done == n:
            elapsed = time.time() - t0
            rate = done / max(elapsed, 1e-6)
            eta = (n - done) / max(rate, 1e-6)
            print(f"  [{label}] {done}/{n} ({rate:.2f} seq/s | ETA: {eta/60:.1f}m)", flush=True)

    return rewards.reshape(n_prompts, 6)  # cols 0..2 = chosen, cols 3..5 = rejected

def compute_rmbench_metrics(scores_6: np.ndarray) -> Dict[str, float]:
    c = scores_6[:, :3]  # (N, 3)
    r = scores_6[:, 3:]  # (N, 3)
    # Official 3x3 style matrix
    easy   = np.mean([(c[:, 1] > r[:, 0]).mean(), (c[:, 2] > r[:, 0]).mean(), (c[:, 2] > r[:, 1]).mean()]) * 100.0
    normal = np.mean([(c[:, 0] > r[:, 0]).mean(), (c[:, 1] > r[:, 1]).mean(), (c[:, 2] > r[:, 2]).mean()]) * 100.0
    hard   = np.mean([(c[:, 0] > r[:, 1]).mean(), (c[:, 0] > r[:, 2]).mean(), (c[:, 1] > r[:, 2]).mean()]) * 100.0
    overall = (easy + normal + hard) / 3.0
    return {
        "easy_pct": round(float(easy), 2),
        "normal_pct": round(float(normal), 2),
        "hard_pct": round(float(hard), 2),
        "overall_pct": round(float(overall), 2),
        "style_gap_easy_minus_hard": round(float(easy - hard), 2),
    }

[RM-BENCH] Downloading THU-KEG/RM-Bench...


README.md:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

total_dataset.json: reconstructing file:   0%|          |  0.00B / 11.0MB            

total_dataset.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1327 [00:00<?, ? examples/s]

[RM-BENCH] Loaded 1327 prompts (7962 total sequences).
[TOKENIZING] Pre-tokenizing all RM-Bench sequences once...
[TOKENIZING] Done (7962 sequences).


In [6]:
# ------------------------------------------------------------------------------
# 5. EVALUATE ALL 4 CHECKPOINTS & SAVE INCREMENTALLY
# ------------------------------------------------------------------------------
OUT_PATH = "/kaggle/working/rmbench_4way_results.json"
all_results = {}

for mode_name, cfg in CHECKPOINTS.items():
    print(f"\n{'='*80}\n[RM-BENCH AUDIT] {mode_name}\n{'='*80}", flush=True)
    set_peft_model_state_dict(
        peft_backbone,
        torch.load(os.path.join(cfg["dir"], "lora_weights.pt"), map_location="cpu", weights_only=False),
    )
    rm = Qwen3AssociativeRM(
        peft_backbone=peft_backbone,
        associative=cfg["associative"],
        use_monotonic=cfg["use_monotonic"],
        use_lrsv_veto=cfg["use_lrsv_veto"],
        scalar_only_mode=cfg["scalar_only_mode"],
        dag_num_layers=2 if cfg["use_dag_mixer"] else 0,
    )
    rm.load_head_state_dict(
        torch.load(os.path.join(cfg["dir"], "arm_heads.pt"), map_location=HEAD_DEVICE, weights_only=False)
    )
    rm.eval()

    scores_6 = score_all_rmbench(rm, all_seqs, label=mode_name)
    metrics = compute_rmbench_metrics(scores_6)
    all_results[mode_name] = metrics
    print(json.dumps(metrics, indent=2), flush=True)

    # Save after every model so nothing is lost on timeout
    with open(OUT_PATH, "w") as f:
        json.dump(all_results, f, indent=2)

    del rm
    torch.cuda.empty_cache()
    gc.collect()


[RM-BENCH AUDIT] SCALAR_RM
  [SCALAR_RM] 4/7962 (4.79 seq/s | ETA: 27.7m)
  [SCALAR_RM] 404/7962 (16.53 seq/s | ETA: 7.6m)
  [SCALAR_RM] 804/7962 (12.86 seq/s | ETA: 9.3m)
  [SCALAR_RM] 1204/7962 (10.86 seq/s | ETA: 10.4m)
  [SCALAR_RM] 1604/7962 (9.37 seq/s | ETA: 11.3m)
  [SCALAR_RM] 2004/7962 (7.18 seq/s | ETA: 13.8m)
  [SCALAR_RM] 2404/7962 (5.71 seq/s | ETA: 16.2m)
  [SCALAR_RM] 2804/7962 (4.77 seq/s | ETA: 18.0m)
  [SCALAR_RM] 3204/7962 (4.12 seq/s | ETA: 19.3m)
  [SCALAR_RM] 3604/7962 (3.66 seq/s | ETA: 19.9m)
  [SCALAR_RM] 4004/7962 (3.28 seq/s | ETA: 20.1m)
  [SCALAR_RM] 4404/7962 (3.00 seq/s | ETA: 19.8m)
  [SCALAR_RM] 4804/7962 (2.78 seq/s | ETA: 19.0m)
  [SCALAR_RM] 5204/7962 (2.59 seq/s | ETA: 17.7m)
  [SCALAR_RM] 5604/7962 (2.42 seq/s | ETA: 16.2m)
  [SCALAR_RM] 6004/7962 (2.28 seq/s | ETA: 14.3m)
  [SCALAR_RM] 6404/7962 (2.14 seq/s | ETA: 12.1m)
  [SCALAR_RM] 6804/7962 (2.02 seq/s | ETA: 9.6m)
  [SCALAR_RM] 7204/7962 (1.90 seq/s | ETA: 6.7m)
  [SCALAR_RM] 7604/7962 (1.7

In [7]:
# ------------------------------------------------------------------------------
# 6. PRINT FINAL COMPARISON TABLE
# ------------------------------------------------------------------------------
print("\n\n### 4-WAY RM-BENCH (OUT-OF-DISTRIBUTION STYLE TRAPS) SUMMARY")
print("| Model | Overall | Easy | Normal | Hard (Style Trap) | Style Gap (Easy - Hard) |")
print("| :--- | :---: | :---: | :---: | :---: | :---: |")
for name, r in all_results.items():
    print(
        f"| **{name}** | **{r['overall_pct']}%** | {r['easy_pct']}% | "
        f"{r['normal_pct']}% | **{r['hard_pct']}%** | {r['style_gap_easy_minus_hard']:.2f} pp |"
    )
print(f"\n[SAVED] {OUT_PATH}")



### 4-WAY RM-BENCH (OUT-OF-DISTRIBUTION STYLE TRAPS) SUMMARY
| Model | Overall | Easy | Normal | Hard (Style Trap) | Style Gap (Easy - Hard) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **SCALAR_RM** | **73.18%** | 86.08% | 75.91% | **57.55%** | 28.54 pp |
| **INDEPENDENT_6D** | **73.64%** | 84.75% | 76.14% | **60.04%** | 24.72 pp |
| **STELLAR_NO_VETO** | **73.83%** | 84.88% | 75.81% | **60.81%** | 24.06 pp |
| **STELLAR_FULL** | **74.31%** | 82.74% | 75.91% | **64.28%** | 18.46 pp |

[SAVED] /kaggle/working/rmbench_4way_results.json
